<a href="https://colab.research.google.com/github/ViniciusPolachini/ReconhecimentoDeCancer/blob/main/Atividade_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Introdução
Esse trabalho visa fazer a classificação de tipos de cancer de pele fazendo uso de *machine learning* para visão computacional.
Para isso, foram usadas técnicas de *transfer learning* e *fine-tuning* para conseguir adequar modelos já pronto para o trabalho.
Foram feitos testes com os modelos InceptionV3, VGG e MobileNetv2. Dos 3, tendo em vista os contextos de execução do trabalho, o MobileNetV2 acabou sendo o escolhido final.
Já referente a base de dados para treinamento, foi utilizado o *Skin Cancer MNIST: HAM10000*
Como resultado, foi possível alcançar um nível de acurácia satisfatório usando o MobileNetV2, na casa dos 78%.

# Materiais
## Ambiente de trabalho
Para execução desse trabalho, foram utilizados dois ambientes de notebook python, o Google Kolab e o Kaggle notebook. O motivo disso foi por conta de limitações de uso de GPU em ambos, em que ao acabar o uso gratuito de um, o outro passava a ser utilizado.
## Base de imagens
Foi utilizado o  *Skin Cancer MNIST: HAM10000*, o qual está presente no kaagle e possui uma base de dados para 7 tipos de lesões de pele diferentes, as quais serão classificadas pelo modelo
## Modelo
Três modelos foram avaliados para serem usados, o VGG, InceptionV3 e MobileNetV2. Ao realizar testes nos 3, 3 critérios foram separados para a decisão final do modelo, que são: tempo de treino e acuracia.
O VGG tinha um tempo de treino razoavel, mas um tamanho grande e estava tendo problemas na acuracia e loss de validação, dando a entender que sempre caia em um padrão de chutes aleatórios ou de neuronios especificos tendo mais peso. Além disso, fazia um grande uso de RAM no colab.
Já o IncecptionV3 demonstrou mais potencial na parte de acuracia, principalmente por conta de sua forma de funcionamento o qual da atenção aos detalhes tanto em maior escala quanto em menor. Mas o tempo de treinamento era longo de mais e ele também era grande.
O MobileNetV2 acabou sendo escolhido tanto por demonstrar uma acuracia satisfatória, quanto por ter um tempo de treinamento mais aceitável e não causar problemas no uso de RAM.
Talvez os outros dois modelos também pudessem ter sido usados melhor se eu aplicasse certas configurações que acabei aplicando no MobileNetV2 e mudanças de como realizava a obtenção de imagens. Mas o tempo de processamento tirava parte do tempo para esses refinamentos.
# Técnicas aplicadas
## Dados de treinamento e avaliação
Para pegar os dados de treinamento e avaliação, foi pego o HAM10000_METADATA.csv, o qual é o dataframe do projeto. Ao acessar ele com pandas, foi adicionando os campos path, o qual era o caminho para as imagens dentro do projeto e o label, o qual era o valor dx mas assegurado como string. Com isso, foram gerados outros dois data frames, um para treino e outro para validação, com o de treino com 8000 imagens e o de validação com 2015 imagens. Por fim, foi utilizado o recurso Image_Generetor do TensorFlow.Keras para gerar dois geradores, um de treinamento e outro de validação. Essa abordagem foi usada para evitar o alto consumo de RAM ao abrir e salvar dados de várias imagens dentro de uma variável.
## Transfer Learning
Com o modelo MobileNetV2 pego, foi retirado sua camada de saída e trocada por uma nova, sendo um Dense com 7 saídas. Além da adição do dense, também foi incorporado um dropout antes da saída para resolver alguns problemas com valores de loss muito altos nos outputs de validação. Após isso, as camadas do MobileNetV2 tem seu treinamento bloqueado e a nova camada tem ele ativo. Com essa configuração, foi feito o primeiro treinamento, com 10 epochs, para configurar a nova camada de saída.
## Fine-tuning
Após o treinamento da camada de saída, esta camada teve seu treinamento bloqueado também. Com ela bloqueada, as camadas do block_16 tiveram seu treinamento habilitado, menos as de BatchNormalization. Antes de começar o treino novamente, o training_rate foi modificado para 0.00005. Com essa configuração, foi feito mais um treino com 10 epochs
# Resultados
Ao final do treinamento, foi alcançado na última epoch uma acuracia de treinamento de 80% e um loss de 0.52. Já na validação, a acuracia ficou em 79% e o loss em 0.59.
Fazendo alguns testes manuais, há falhas de classificação, principalmente em classes com menos imagens, ou classes que tem padrões visuais que podem ser parecidos, como ml e nv






In [ ]:
import os
import zipfile
import kagglehub
import pandas as pd
import keras
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from glob import glob
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Activation
from keras.layers import Conv2D, MaxPooling2D
from sklearn.model_selection import train_test_split
from keras.models import Model

In [ ]:
#Download das imagens
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path: ", path)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Path:  /kaggle/input/skin-cancer-mnist-ham10000


In [ ]:
df = pd.read_csv(path+"/HAM10000_metadata.csv")

all_image_paths = {os.path.basename(x).split('.')[0]: x
                   for x in glob(os.path.join(path, '*', '*.jpg'))}

df['path'] = df['image_id'].map(all_image_paths)
df['label'] = df['dx'].astype(str)

df_treino, df_validacao = train_test_split(
    df,
    train_size=8000,
    stratify=df['label'],
    random_state=42
)

print(f"Total para Treino: {len(df_treino)}")
print(f"Total para Validação: {len(df_validacao)}")
contagem = df['label'].value_counts()

print("Quantidade por classe:")
print(contagem)

Total para Treino: 8000
Total para Validação: 2015
Quantidade por classe:
label
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_generator = datagen.flow_from_dataframe(
    dataframe=df_treino,
    x_col="path",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_generator = datagen.flow_from_dataframe(
    dataframe=df_validacao,
    x_col="path",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)
print(train_generator.class_indices)

Found 8000 validated image filenames belonging to 7 classes.
Found 2015 validated image filenames belonging to 7 classes.
{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


In [ ]:
model = keras.applications.MobileNetV2();
model.summary()

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 3,538,984 (13.50 MB)

 Trainable params: 3,504,872 (13.37 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [ ]:
inp = model.input

num_classes = df_treino['label'].nunique();

x = model.layers[-2].output
x = Dropout(0.5)(x)
out = Dense(num_classes, activation='softmax')(x)

model_new = Model(inp, out)

In [ ]:
for l, layer in enumerate(model_new.layers[:-1]):
    layer.trainable = False

model_new.layers[-1].trainable = True

model_new.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

for l, layer in enumerate(model_new.layers):
    if layer.trainable is True:
        print(layer.name)

model_new.summary()

dense


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model_new.fit(train_generator,
                         epochs=10,
                       validation_data=val_generator)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 175s 606ms/step - accuracy: 0.6568 - loss: 1.0903 - val_accuracy: 0.7419 - val_loss: 0.7442
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 71s 282ms/step - accuracy: 0.7157 - loss: 0.8238 - val_accuracy: 0.7489 - val_loss: 0.7056
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 278ms/step - accuracy: 0.7331 - loss: 0.7475 - val_accuracy: 0.7524 - val_loss: 0.6879
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 70s 278ms/step - accuracy: 0.7389 - loss: 0.7178 - val_accuracy: 0.7767 - val_loss: 0.6593
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 278ms/step - accuracy: 0.7496 - loss: 0.6886 - val_accuracy: 0.7663 - val_loss: 0.6667
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 276ms/step - accuracy: 0.7498 - loss: 0.6879 - val_accuracy: 0.7608 - val_loss: 0.6689
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 277ms/step - accuracy: 0.7509 - loss: 0.6831 - val_accuracy: 0.7653 - val_loss: 0.6666
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 71s 284ms/step - accuracy: 0.7595 - loss: 

In [ ]:
for layer in model_new.layers:
    layer.trainable = False

for l, layer in enumerate(model_new.layers):
    if ('block_16' in layer.name) :
      if 'BN' not in layer.name and 'BatchNormalization' not in layer.name:
        print(layer.name)
        layer.trainable = True

model_new.layers[-1].trainable = False

model_new.compile(loss='categorical_crossentropy',
              optimizer=Adam(1e-5),
              metrics=['accuracy'])
print('\n')
for l, layer in enumerate(model_new.layers):
    if layer.trainable is True:
        print(layer.name)

model_new.summary(line_length=150)

block_16_expand
block_16_expand_relu
block_16_depthwise
block_16_depthwise_relu
block_16_project


block_16_expand
block_16_expand_relu
block_16_depthwise
block_16_depthwise_relu
block_16_project


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━
┃ Layer (type)                               ┃ Output Shape                         ┃                 Param # ┃ Con
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━
│ input_layer (InputLayer)                   │ (None, 224, 224, 3)                  │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ Conv1 (Conv2D)                             │ (None, 112, 112, 32)                 │                     864 │ inp
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ bn_Conv1 (BatchNormalization)              │ (None, 112, 112, 32)                 │                     128 │ Con
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ Conv1_relu (ReLU)                          │ (None, 112, 112, 32)                 │                       0 │ bn_
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ expanded_conv_depthwise (DepthwiseConv2D)  │ (None, 112, 112, 32)                 │                     288 │ Con
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ expanded_conv_depthwise_BN                 │ (None, 112, 112, 32)                 │                     128 │ exp
│ (BatchNormalization)                       │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ expanded_conv_depthwise_relu (ReLU)        │ (None, 112, 112, 32)                 │                       0 │ exp
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ expanded_conv_project (Conv2D)             │ (None, 112, 112, 16)                 │                     512 │ exp
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ expanded_conv_project_BN                   │ (None, 112, 112, 16)                 │                      64 │ exp
│ (BatchNormalization)                       │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ block_1_expand (Conv2D)                    │ (None, 112, 112, 96)                 │                   1,536 │ exp
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ block_1_expand_BN (BatchNormalization)     │ (None, 112, 112, 96)                 │                     384 │ blo
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ block_1_expand_relu (ReLU)                 │ (None, 112, 112, 96)                 │                       0 │ blo
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ block_1_pad (ZeroPadding2D)                │ (None, 113, 113, 96)                 │                       0 │ blo
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ block_1_depthwise (DepthwiseConv2D)        │ (None, 56, 56, 96)                   │                     864 │ blo
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ block_1_depthwise_BN (BatchNormalization)  │ (None, 56, 56, 96)                   │                     384 │ blo
├────────────────────────────────────────────┼──────────

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 469,440 (1.79 MB)

 Non-trainable params: 1,797,511 (6.86 MB)

In [ ]:
history2 = model_new.fit(train_generator,
                         epochs=10,
                       validation_data=val_generator)


Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 87s 310ms/step - accuracy: 0.7719 - loss: 0.6221 - val_accuracy: 0.7792 - val_loss: 0.6275
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 70s 279ms/step - accuracy: 0.7794 - loss: 0.5956 - val_accuracy: 0.7826 - val_loss: 0.6311
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 290ms/step - accuracy: 0.7819 - loss: 0.5865 - val_accuracy: 0.7737 - val_loss: 0.6228
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 71s 283ms/step - accuracy: 0.7844 - loss: 0.5744 - val_accuracy: 0.7826 - val_loss: 0.6113
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 71s 284ms/step - accuracy: 0.7891 - loss: 0.5709 - val_accuracy: 0.7836 - val_loss: 0.6022
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 70s 278ms/step - accuracy: 0.7930 - loss: 0.5614 - val_accuracy: 0.7896 - val_loss: 0.5988
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 70s 281ms/step - accuracy: 0.8001 - loss: 0.5487 - val_accuracy: 0.7975 - val_loss: 0.5908
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 70s 281ms/step - accuracy: 0.7976 - loss: 0

In [ ]:
def get_image(path):
    img = image.load_img(path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_pre_process = preprocess_input(img_array)
    return img_pre_process

In [ ]:
df_shuffle = df.sample(frac=1).reset_index(drop=True)
amostras_por_classe = df_shuffle.drop_duplicates(subset='label')

for index, row in amostras_por_classe.iterrows():
    print(f"Doença: {row['dx']} | Path: {row['path']}")

Doença: nv | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_2/ISIC_0031559.jpg
Doença: akiec | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0024925.jpg
Doença: bcc | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0028197.jpg
Doença: df | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0026254.jpg
Doença: vasc | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0025873.jpg
Doença: bkl | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0027026.jpg
Doença: mel | Path: /kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_2/ISIC_0034284.jpg


In [ ]:
def classifiesCancer(path, expected):
    image = get_image(path)
    predict = model_new.predict(image)

    classe_predita = np.argmax(predict)
    confianca = np.max(predict) * 100

    mapeamento_doencas = {
        0: 'Aknitic keratoses (akiec)',
        1: 'Basal cell carcinoma (bcc)',
        2: 'Benign keratosis-like lesions (bkl)',
        3: 'Dermatofibroma (df)',
        4: 'Melanoma (mel)',
        5: 'Melanocytic nevi (nv)',
        6: 'Vascular lesions (vasc)'
    }

    resultado_final = mapeamento_doencas[classe_predita]

    print(f"Esperado: {expected}")
    print(f"Classe detectada: {classe_predita} com {confianca:.2f}% de confiança.")
    print(f"O diagnóstico do modelo é: {resultado_final}")

In [ ]:
for index, row in amostras_por_classe.iterrows():
    classifiesCancer(row['path'], row['label'])

'''
Quantidade de amostras para cada tipo
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
'''

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
Esperado: nv
Classe detectada: 5 com 97.81% de confiança.
O diagnóstico do modelo é: Melanocytic nevi (nv)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Esperado: akiec
Classe detectada: 1 com 49.48% de confiança.
O diagnóstico do modelo é: Basal cell carcinoma (bcc)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Esperado: bcc
Classe detectada: 1 com 42.71% de confiança.
O diagnóstico do modelo é: Basal cell carcinoma (bcc)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Esperado: df
Classe detectada: 5 com 41.18% de confiança.
O diagnóstico do modelo é: Melanocytic nevi (nv)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Esperado: vasc
Classe detectada: 6 com 96.71% de confiança.
O diagnóstico do modelo é: Vascular lesions (vasc)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Esperado: bkl
Classe detectada: 4 com 34.30% de confiança.
O diagnóstico do modelo é: Melanoma (mel)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Esperado: mel
Classe detectada: 5 com 53.24% de confiança.
O diagnóstico do mode

'\nQuantidade de amostras para cada tipo\nnv       6705\nmel      1113\nbkl      1099\nbcc       514\nakiec     327\nvasc      142\ndf        115\n'